# Panel Data Assumption Tests with panelbox

**Series:** panelbox Validation Tutorials | **Notebook:** 01 of 04  
**Level:** Intermediate | **Estimated time:** 90–120 minutes

---

## Motivation

Before reporting econometric results, always check that model assumptions hold — violations lead to **invalid standard errors** and **unreliable inference**. Even when the coefficient estimates themselves are unbiased, standard errors can be severely underestimated, causing you to find "significant" effects that do not actually exist.

This notebook walks through three families of assumption violations that are especially common in panel data:

| Family | What it means | Consequence |
|--------|--------------|-------------|
| **Heteroskedasticity** | Error variance differs across entities or time | Biased standard errors |
| **Serial correlation** | Errors within an entity are correlated over time | Underestimated SEs → inflated t-stats |
| **Cross-sectional dependence** | Errors of different entities are correlated at the same time | Underestimated SEs → anti-conservative inference |

## Roadmap

```
Detect a problem  →  Run the appropriate test  →  Apply the correct correction
```

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why assumption violations matter (bias vs. efficiency vs. inference)
2. Run heteroskedasticity tests (Modified Wald, Breusch-Pagan, White)
3. Run serial correlation tests (Wooldridge AR, Breusch-Godfrey, Baltagi-Wu)
4. Run cross-sectional dependence tests (Pesaran CD, BP LM, Frees)
5. Use `ValidationSuite` as a unified pre-publication routine
6. Choose the correct robust standard error correction after diagnosis

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import pathlib

# Make panelbox and local utils importable
ROOT = pathlib.Path("..").resolve()          # examples/validation/
PANELBOX_ROOT = pathlib.Path("../../..")
for p in [str(ROOT), str(PANELBOX_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

# Dataset loading
from panelbox.datasets import load_dataset

# Local utility functions
from utils.plot_helpers import (
    plot_residuals_by_entity,
    plot_acf_panel,
    plot_correlation_heatmap,
)

# panelbox models
from panelbox.models.static import FixedEffects, PooledOLS, RandomEffects

# panelbox validation
from panelbox.validation import ValidationSuite
from panelbox.validation.heteroskedasticity import (
    ModifiedWaldTest,
    BreuschPaganTest,
    WhiteTest,
)
from panelbox.validation.serial_correlation import (
    WooldridgeARTest,
    BreuschGodfreyTest,
    BaltagiWuTest,
)
from panelbox.validation.cross_sectional_dependence import (
    PesaranCDTest,
    BreuschPaganLMTest,
    FreesTest,
)

# Output directory
OUTPUT_DIR = pathlib.Path("..") / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("All imports OK")

---

## Section 1 — Why Assumptions Matter

### 1.1 The Five Classical Panel Data Assumptions

The Gauss-Markov framework requires five conditions for OLS to be BLUE (Best Linear Unbiased Estimator). In panel settings these conditions carry over, but they can fail in systematic ways:

1. **Linearity** — the relationship between y and X is linear in parameters.  
   *Violation*: coefficient estimates β̂ are biased.

2. **Strict exogeneity** — E[εᵢₜ | Xᵢ₁, …, XᵢT] = 0.  
   *Violation*: coefficient estimates β̂ are biased (e.g., omitted variable bias).

3. **Homoskedasticity** — Var(εᵢₜ | X) = σ² (constant for all i, t).  
   *Violation*: SEs are wrong, but β̂ is still unbiased.

4. **No serial correlation** — Cov(εᵢₜ, εᵢₛ | X) = 0 for t ≠ s.  
   *Violation*: SEs are underestimated, β̂ is still unbiased.

5. **Normality** (for exact finite-sample inference) — εᵢₜ | X ~ N(0, σ²).  
   *Violation*: t and F distributions only hold approximately in large samples.

### Consequence Table

| Violation | β̂ biased? | SE biased? | Inference valid? |
|-----------|-----------|-----------|------------------|
| Heteroskedasticity | **No** | **Yes** | **No** |
| Serial correlation | **No** | **Yes** (underest.) | **No** |
| Cross-sectional dependence | **No** | **Yes** (underest.) | **No** |
| Endogeneity / omitted variable | **Yes** | **Yes** | **No** |

> **Key takeaway:** All three violations in this notebook leave your point estimates β̂ unbiased but corrupt your standard errors. You may be drawing completely wrong inference (rejecting true H₀) without knowing it.

### 1.2 Load and Inspect Datasets

In [ ]:
df_firms = load_dataset("firmdata", category="validation")
df_macro = load_dataset("macro_panel", category="validation")

print("=== firmdata ===")
print(f"Shape: {df_firms.shape}")
print(df_firms.head())
print("\nsize_category distribution:")
print(df_firms["size_category"].value_counts())

print("\n=== macro_panel ===")
print(f"Shape: {df_macro.shape}")
print(df_macro.head())

**Expected output:** `firmdata` shape (1000, 6) with columns `firm_id, year, sales, capital, labor, size_category`; `macro_panel` shape (600, 6) with columns `country, year, gdp_growth, investment, trade_openness, region`.

The `size_category` variable (small / medium / large) is the key to the groupwise heteroskedasticity we will detect — larger firms have higher residual variance by construction.

---

## Section 2 — Heteroskedasticity Tests

### 2.1 Groupwise Heteroskedasticity in Panels

**Groupwise heteroskedasticity** arises when each entity has its own error variance:  
Var(εᵢₜ) = σ²ᵢ  (different for each entity i).

This is extremely common in firm-level panels: large firms are simply more volatile than small ones.

#### Modified Wald Test

- **H₀:** σ²₁ = σ²₂ = ... = σ²ₙ (all entity variances equal)
- **H₁:** at least one entity has a different variance
- **Statistic:** W = Σᵢ Tᵢ · ln(σ̂²_pooled / σ̂²ᵢ), distributed χ²(N) under H₀
- Designed specifically for **Fixed Effects** models; robust to serial correlation.

#### Which test to use?

| Test | Best for | Null hypothesis |
|------|----------|-----------------|
| **Modified Wald** | Fixed Effects models | Groupwise homoskedasticity |
| **Breusch-Pagan** | All panel models | Var(ε) does not depend on X |
| **White** | All models, no functional form assumption | General homoskedasticity |

### 2.2 Fit Baseline Fixed Effects Model

In [ ]:
fe_firms = FixedEffects(
    data=df_firms,
    formula="sales ~ capital + labor",
    entity_col="firm_id",
    time_col="year",
).fit()

print(fe_firms.summary())

### 2.3 Modified Wald Test

In [ ]:
mwald = ModifiedWaldTest(fe_firms)
result_mwald = mwald.run(alpha=0.05)

print(result_mwald)
print(f"Test statistic : {result_mwald.statistic:.4f}")
print(f"p-value        : {result_mwald.pvalue:.4f}")
print(f"Reject H\u2080      : {result_mwald.reject_null}")
print(f"Variance ratio : {result_mwald.metadata['variance_ratio']:.2f}")

**Interpretation:**

- If we **reject H₀** (p < 0.05), there is statistical evidence of groupwise heteroskedasticity. The variance ratio tells us how severe it is:
  - Ratio < 2: Minor — robust SEs optional
  - Ratio 2–10: Moderate — use White or cluster-robust SEs
  - Ratio > 10: Severe — consider FGLS
- **Correction:** Re-fit the model with `cov_type="robust"` (White) or `cov_type="clustered"` (entity-cluster-robust).

### 2.4 Breusch-Pagan Test

In [ ]:
bp = BreuschPaganTest(fe_firms)
result_bp = bp.run(alpha=0.05)
print(result_bp)

### 2.5 White Test

In [ ]:
white = WhiteTest(fe_firms)
result_white = white.run(alpha=0.05)
print(result_white)

### 2.6 Visualization — Residuals by Firm Size

In [ ]:
# Align size_category with residuals by matching on firm_id and year index
resid_series = pd.Series(fe_firms.resid, name="resid")
entity_series = pd.Series(fe_firms.entity_index, name="firm_id")

fig = plot_residuals_by_entity(
    resid=resid_series,
    entity_col=entity_series,
    title="Residuals by Firm (sorted by median residual)",
    max_entities=20,
)
fig.suptitle("Residuals by Firm — Detecting Groupwise Heteroskedasticity", fontsize=13, y=1.02)
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "01_residual_boxplot.png", dpi=100, bbox_inches="tight")
print("Saved: outputs/01_residual_boxplot.png")
plt.show()

> **Expected pattern:** The interquartile range of the boxes should visually widen for large firms, confirming that the Modified Wald and BP tests rejected H₀.

### 2.7 Interpretation and Correction

All three tests should reject the null of homoskedasticity on `firmdata` because the data were generated with `σ_large = 4 × σ_small`.

**Correction options:**

| Correction | When to use | panelbox argument |
|------------|-------------|-------------------|
| White (HC) robust SEs | Heteroskedasticity only | `cov_type="robust"` |
| Entity-cluster robust SEs | Heteroskedasticity + serial correlation | `cov_type="clustered"` |
| Driscoll-Kraay SEs | Heteroskedasticity + serial + CD | `cov_type="driscoll-kraay"` |

Let's compare naive vs. cluster-robust standard errors:

In [ ]:
fe_robust = FixedEffects(
    data=df_firms,
    formula="sales ~ capital + labor",
    entity_col="firm_id",
    time_col="year",
).fit(cov_type="clustered")

comparison = pd.DataFrame({
    "coef_naive"  : fe_firms.params,
    "se_naive"    : fe_firms.std_errors,
    "t_naive"     : fe_firms.tvalues,
    "se_cluster"  : fe_robust.std_errors,
    "t_cluster"   : fe_robust.tvalues,
})
print("=== Naive vs. Cluster-Robust Standard Errors ===")
print(comparison.round(4))

---

## Section 3 — Serial Correlation Tests

### 3.1 Serial Correlation in Panel Data

**Serial (autocorrelation)** means that errors for the same entity are correlated across time:

**AR(1) model:** εᵢₜ = ρ · εᵢ,ₜ₋₁ + νᵢₜ, where |ρ| < 1

**Consequences of ignoring serial correlation:**
- OLS standard errors are *underestimated* (sometimes severely)
- t-statistics are *inflated* → Type I error increases (you reject H₀ too often)
- Confidence intervals are *too narrow*

#### Available tests

| Test | Key feature | Null hypothesis |
|------|-------------|-----------------|
| **Wooldridge AR** | Simple first-difference approach; robust to heteroskedasticity; requires T ≥ 3 | No first-order serial correlation |
| **Breusch-Godfrey** | Flexible; tests AR(p) of any order; based on auxiliary regression | No serial correlation up to order p |
| **Baltagi-Wu LBI** | Locally Best Invariant; values near 2 = no autocorrelation (like Durbin-Watson for panels) | No serial correlation |

### 3.2 Fit Fixed Effects on Macro Panel

In [ ]:
fe_macro = FixedEffects(
    data=df_macro,
    formula="gdp_growth ~ investment + trade_openness",
    entity_col="country",
    time_col="year",
).fit()

print(fe_macro.summary())

### 3.3 Wooldridge AR Test

In [ ]:
wooldridge = WooldridgeARTest(fe_macro)
result_w = wooldridge.run(alpha=0.05)

print(result_w)
print(f"F-statistic : {result_w.statistic:.4f}")
print(f"p-value     : {result_w.pvalue:.4f}")
print(f"Reject H\u2080   : {result_w.reject_null}")

### 3.4 Breusch-Godfrey Test (lags 1 and 2)

In [ ]:
bg = BreuschGodfreyTest(fe_macro)

result_bg1 = bg.run(lags=1, alpha=0.05)
result_bg2 = bg.run(lags=2, alpha=0.05)

print("Breusch-Godfrey AR(1):")
print(result_bg1)

print("\nBreusch-Godfrey AR(2):")
print(result_bg2)

### 3.5 Baltagi-Wu LBI Test

In [ ]:
bw = BaltagiWuTest(fe_macro)
result_bw = bw.run(alpha=0.05)

print(result_bw)
print(f"LBI statistic : {result_bw.statistic:.4f}")
print("(Values near 2 indicate no autocorrelation; values < 2 suggest positive autocorrelation)")

### 3.6 Visualization — ACF of Residuals

In [ ]:
resid_macro = pd.Series(fe_macro.resid)
entity_macro = pd.Series(fe_macro.entity_index)

fig = plot_acf_panel(
    resid=resid_macro,
    entity_col=entity_macro,
    lags=8,
    n_sample=6,
    title="ACF of Residuals — Sample of Countries",
)
fig.suptitle("ACF of Residuals — Sample of Countries", fontsize=13, y=1.02)
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "01_acf_panel.png", dpi=100, bbox_inches="tight")
print("Saved: outputs/01_acf_panel.png")
plt.show()

> **Expected pattern:** The bar at lag 1 should exceed the red dashed 95% confidence band for most countries, confirming significant AR(1) autocorrelation.

### 3.7 Correction for Serial Correlation

When serial correlation is detected:

- **HAC (Newey-West)** SEs correct for serial correlation within entities.
- **Driscoll-Kraay** SEs (recommended for macro panels) correct for both serial correlation *and* cross-sectional dependence simultaneously.

```python
fe_hac = FixedEffects(
    data=df_macro,
    formula="gdp_growth ~ investment + trade_openness",
    entity_col="country",
    time_col="year",
).fit(cov_type="driscoll-kraay")
```

---

## Section 4 — Cross-Sectional Dependence Tests

### 4.1 What is Cross-Sectional Dependence?

**Cross-sectional dependence (CD)** means that residuals of different entities at the *same* time period are correlated:

Cov(εᵢₜ, εⱼₜ) ≠ 0 for i ≠ j

**Why does this happen?**
- **Common shocks** — the 2008 global financial crisis, oil price spikes, pandemics — affect all countries simultaneously
- **Spatial spillovers** — neighboring regions may share economic cycles
- **Omitted global factors** — a missing worldwide trend absorbed into residuals

**Consequences:**
- Standard errors are *underestimated* → anti-conservative inference
- Pooled estimators may be inconsistent if the CD is correlated with regressors

**When CD matters most:** macro panels with N < T (countries, regions), or any panel with global shocks.

#### Available tests

| Test | Best for | Distribution |
|------|----------|--------------|
| **Pesaran CD** | Large N, small T | N(0,1) |
| **Breusch-Pagan LM** | Small to moderate N (N < 30) | χ²(N(N-1)/2) |
| **Frees** | Non-parametric; robust to non-normality | Frees Q distribution |

### 4.2 Pesaran CD Test

In [ ]:
pesaran = PesaranCDTest(fe_macro)
result_cd = pesaran.run(alpha=0.05)

print(result_cd)
print(f"CD statistic       : {result_cd.statistic:.4f}")
print(f"Average correlation: {result_cd.metadata['avg_correlation']:.4f}")
print(f"Avg |correlation|  : {result_cd.metadata['avg_abs_correlation']:.4f}")
print(f"p-value            : {result_cd.pvalue:.4f}")
print(f"Reject H\u2080          : {result_cd.reject_null}")

### 4.3 Breusch-Pagan LM Test

> **Note:** The BP LM test is appropriate when N is small to moderate (N < 30). For our macro panel with N = 30 countries it is borderline appropriate; Pesaran CD is preferred for larger N.

In [ ]:
bp_lm = BreuschPaganLMTest(fe_macro)
result_bplm = bp_lm.run(alpha=0.05)
print(result_bplm)

### 4.4 Frees Test

In [ ]:
frees = FreesTest(fe_macro)
result_frees = frees.run(alpha=0.05)
print(result_frees)

### 4.5 Visualization — Residual Correlation Heatmap

In [ ]:
# Build the cross-entity correlation matrix directly to avoid pivot alignment issues
resid_vals = np.asarray(fe_macro.resid).ravel()
entity_vals = np.asarray(fe_macro.entity_index)

resid_df_cd = pd.DataFrame({"residual": resid_vals, "entity": entity_vals})
resid_df_cd["t"] = resid_df_cd.groupby("entity").cumcount()   # time index within entity

# Pivot: rows = time, columns = entity
wide_cd = resid_df_cd.pivot(index="t", columns="entity", values="residual")
corr_cd = wide_cd.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_cd.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
labels = [str(e) for e in corr_cd.columns]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_yticklabels(labels, fontsize=7)
ax.set_title("Cross-Entity Residual Correlation Matrix")
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "01_correlation_heatmap.png", dpi=100, bbox_inches="tight")
print("Saved: outputs/01_correlation_heatmap.png")
plt.show()

> **Expected pattern:** Most off-diagonal cells should show positive (red-ish) correlations. A block structure would confirm that countries within the same region share common shocks.

### 4.6 Correction for Cross-Sectional Dependence

The **Driscoll-Kraay** estimator is the workhorse correction for macro panels — it is robust to both serial correlation *and* cross-sectional dependence:

```python
fe_dk = FixedEffects(
    data=df_macro,
    formula="gdp_growth ~ investment + trade_openness",
    entity_col="country",
    time_col="year",
).fit(cov_type="driscoll-kraay", bandwidth=4)
```

The `bandwidth` parameter controls how many lags of serial correlation to correct for (a common rule of thumb: `bandwidth = T^(1/4)`).

---

## Section 5 — ValidationSuite: Unified Interface

### 5.1 Why ValidationSuite?

Running individual tests is educational, but in production you want a **single call** that:

1. Runs all relevant tests for your model type automatically
2. Returns a **structured report** with pass/fail status
3. Shows which tests rejected H₀ and suggests corrections
4. Is reproducible and can be serialised to JSON for audit trails

`ValidationSuite` does exactly this. The `tests` argument accepts:
- `"all"` — every available test
- `"default"` — recommended tests for the model type (FE → serial + het + CD)
- `"serial"`, `"het"`, `"cd"` — single category
- A list of category strings

### 5.2 Run ValidationSuite

In [ ]:
suite = ValidationSuite(fe_macro)
report = suite.run(tests="all", alpha=0.05)

print(report)

### 5.3 Inspect Report Details

In [ ]:
# List tests that rejected H₀
failed = report.get_failed_tests()
print(f"Tests that rejected H\u2080 ({len(failed)} total):")
for t in failed:
    print(f"  - {t}")

# Get summary as a DataFrame
summary_df = report.summary(as_dataframe=True)
print("\nSummary DataFrame:")
print(summary_df.to_string())

# Export report as dict (can be saved to JSON)
import json
report_dict = report.to_dict()

json_path = OUTPUT_DIR / "01_validation_report_macro.json"
with open(json_path, "w") as f:
    # Convert non-serialisable types
    def _make_serialisable(obj):
        if isinstance(obj, dict):
            return {k: _make_serialisable(v) for k, v in obj.items()}
        elif isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        elif isinstance(obj, bool):
            return obj
        else:
            return obj
    json.dump(_make_serialisable(report_dict), f, indent=2)

print(f"\nReport saved to: {json_path}")

### 5.4 ValidationSuite on Multiple Models

In [ ]:
models = {
    "Pooled OLS": PooledOLS(
        data=df_macro,
        formula="gdp_growth ~ investment + trade_openness",
        entity_col="country",
        time_col="year",
    ).fit(),
    "Fixed Effects": fe_macro,
    "Random Effects": RandomEffects(
        data=df_macro,
        formula="gdp_growth ~ investment + trade_openness",
        entity_col="country",
        time_col="year",
    ).fit(),
}

print(f"{'Model':<20} {'Tests run':<12} {'Tests failed':<15} {'Failed list'}")
print("-" * 70)
for name, model in models.items():
    s = ValidationSuite(model)
    r = s.run(tests="default", alpha=0.05)
    n_failed = len(r.get_failed_tests())
    n_total = (
        len(r.specification_tests)
        + len(r.serial_tests)
        + len(r.het_tests)
        + len(r.cd_tests)
    )
    failed_names = ", ".join(r.get_failed_tests()) if n_failed else "none"
    print(f"{name:<20} {n_total:<12} {n_failed:<15} {failed_names}")

---

## Section 6 — Complete Case Study

### 6.1 Scenario

A researcher fits a Fixed Effects model on `firmdata.csv` and initially reports naive (non-robust) standard errors. The goal of this section is to:

1. Diagnose the problems with `ValidationSuite`
2. Apply the correct correction (cluster-robust SEs)
3. Compare naive vs. corrected inference
4. Confirm the corrected model passes validation

**Expected problems:** groupwise heteroskedasticity (size_category effect) and mild serial correlation.

### 6.2 Step 1 — Baseline Fit (Naive SEs)

In [ ]:
fe_naive = FixedEffects(
    data=df_firms,
    formula="sales ~ capital + labor",
    entity_col="firm_id",
    time_col="year",
).fit()

print("=== Baseline FE (Naive SEs) ===")
print(fe_naive.summary())

### 6.3 Step 2 — Run Diagnostics

In [ ]:
suite_firms = ValidationSuite(fe_naive)
report_firms = suite_firms.run(tests="all", alpha=0.05)
print(report_firms)

### 6.4 Step 3 — Apply Cluster-Robust SEs

In [ ]:
fe_cluster = FixedEffects(
    data=df_firms,
    formula="sales ~ capital + labor",
    entity_col="firm_id",
    time_col="year",
).fit(cov_type="clustered")

print("=== FE with Cluster-Robust SEs ===")
print(fe_cluster.summary())

### 6.5 Step 4 — Compare Inference

In [ ]:
comparison = pd.DataFrame({
    "coef_naive"  : fe_naive.params,
    "se_naive"    : fe_naive.std_errors,
    "t_naive"     : fe_naive.tvalues,
    "se_cluster"  : fe_cluster.std_errors,
    "t_cluster"   : fe_cluster.tvalues,
})
print("=== Naive vs. Cluster-Robust Inference ===")
print(comparison.round(4))

# SE inflation ratio
se_ratio = fe_cluster.std_errors / fe_naive.std_errors
print("\nSE ratio (cluster / naive):")
print(se_ratio.round(3))
print(f"\nAverage SE inflation: {se_ratio.mean():.2f}x")

### 6.6 Step 5 — Save Firms Validation Report

In [ ]:
# Save the diagnostic report for the firms model
import json

report_firms_dict = report_firms.to_dict()
json_path_firms = OUTPUT_DIR / "01_validation_report_firms.json"

with open(json_path_firms, "w") as f:
    def _ser(obj):
        if isinstance(obj, dict):
            return {k: _ser(v) for k, v in obj.items()}
        elif isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        elif isinstance(obj, bool):
            return obj
        else:
            return obj
    json.dump(_ser(report_firms_dict), f, indent=2)

print(f"Firms validation report saved to: {json_path_firms}")

# Show which tests failed
failed_firms = report_firms.get_failed_tests()
print(f"\nTests that rejected H\u2080 ({len(failed_firms)}):")
for t in failed_firms:
    print(f"  - {t}")

### 6.7 Summary and Takeaways

In this case study we demonstrated the full diagnostic cycle:

| Step | Action | Result |
|------|--------|--------|
| 1 | Fit FE model with naive SEs | Coefficients correct; SEs likely biased |
| 2 | Run `ValidationSuite` | Detected heteroskedasticity (Modified Wald, BP, White all rejected) |
| 3 | Apply `cov_type="clustered"` | SEs increased (ratio shown above) |
| 4 | Compare t-statistics | Cluster-robust t-stats are smaller but remain significant |
| 5 | Save JSON report | Audit trail for reproducibility |

**Key takeaways:**

1. **Naive SEs underestimated uncertainty** — the SE ratio shows by how much.
2. **After cluster-robust correction, t-statistics decreased** but the economic conclusions held.
3. **Always run `ValidationSuite` before reporting** — it takes seconds and prevents publication errors.
4. **Serial correlation and CD compound heteroskedasticity** — when macro panels show all three, use Driscoll-Kraay.

---

## Glossary

| Term | Definition |
|------|------------|
| **Homoskedasticity** | Constant variance of errors: Var(εᵢₜ) = σ² |
| **Groupwise heteroskedasticity** | Variance differs by entity/group: Var(εᵢₜ) = σ²ᵢ |
| **AR(1)** | First-order autoregressive process: εᵢₜ = ρεᵢ,ₜ₋₁ + νᵢₜ |
| **Cross-sectional dependence** | Correlation of errors across entities at same time: Cov(εᵢₜ, εⱼₜ) ≠ 0 |
| **LM test** | Lagrange multiplier test; based on auxiliary regression of squared residuals |
| **Driscoll-Kraay SE** | Standard errors robust to both cross-sectional dependence and serial correlation |
| **LBI** | Locally Best Invariant (Baltagi-Wu); panel analogue of Durbin-Watson |

---

## Next Steps

- **Notebook 02:** Bootstrap and Cross-Validation for robust inference
- **Notebook 03:** Outlier detection and influence analysis
- **Notebook 04:** Model comparison experiments